# LLM4Rec XAI Analysis

This notebook runs a single-example XAI pass on the current ranking layout.

It does not retrain, rebuild metrics, or re-run evaluation. It only loads the
current checkpoints and writes explanation artifacts for:

- Integrated Gradients
- Layer attributions
- Attention rollout
- ALTI+
- Grad-CAM

The four model variants are wired in, and the LoRA branches are skipped only if
`llama31-1b-movielens-ranking-lora/` is missing.


## 1. Mount Google Drive


In [ ]:
from google.colab import drive

drive.mount('/content/drive')


## 2. Point the notebook at this repo

Change `PROJECT_DIR` if your copy lives somewhere else in Drive.


In [ ]:
import os
import sys
from pathlib import Path

PROJECT_DIR = '/content/drive/MyDrive/ECS172/project-ADI'  # change if needed
os.chdir(PROJECT_DIR)
ROOT = Path(PROJECT_DIR)

print('Current directory:', ROOT)
print('Top-level files:', [p.name for p in sorted(ROOT.iterdir(), key=lambda p: p.name)[:15]])
sys.path.insert(0, str(ROOT))


## 3. Install dependencies


In [ ]:
!pip install -q -r requirements.txt
!pip install -q captum matplotlib seaborn
!pip install -q --upgrade torchao
print('Dependencies installed.')


## 4. Optional Hugging Face login

Only needed if you want to use a gated model. The default `unsloth` base model
works without a token.


In [ ]:
# from huggingface_hub import login
# login('YOUR_HF_TOKEN_HERE')


## 5. Configure the XAI run


In [ ]:
import json
import random

import pandas as pd
import torch
from IPython.display import display

from llm4rec_xai_pipeline import RankingXAIPipeline, available_configs
from src.data import IdMaps
from src.ranking_data import load_ranking_examples

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

torch.set_float32_matmul_precision('high')
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)

CHECKPOINT_DIR = ROOT / 'checkpoints'
OUTPUT_DIR = ROOT / 'xai_outputs'
RUN_ROOT = OUTPUT_DIR / 'single_example_xai'
RUN_ROOT.mkdir(parents=True, exist_ok=True)

BASE_MODEL = 'unsloth/Llama-3.2-1B-Instruct'
LORA_MODEL = ROOT / 'llama31-1b-movielens-ranking-lora'

SAMPLE_SIZE = 1
RANDOM_SEED = 42
TARGET_MODES = ('ground_truth', 'model_choice')
INCLUDE_ATTENTION_MAPS = True
IG_STEPS = 16
LAYER_STEPS = 8
LAYER_ATTRIBUTION_METHOD = 'lig'


## 6. Load the test examples


In [ ]:
id_maps = IdMaps.from_json(CHECKPOINT_DIR / 'id_maps.json')
examples = load_ranking_examples(
    ROOT / 'test_ranking_prompts.json',
    ROOT / 'test_ranking.csv',
    id_maps,
    n_history=10,
)

rng = random.Random(RANDOM_SEED)
if SAMPLE_SIZE < len(examples):
    examples = rng.sample(examples, SAMPLE_SIZE)

print(f'Loaded {len(examples)} example(s) for XAI.')
preview = pd.DataFrame([
    {
        'user_id': ex.user_id,
        'true_position': ex.true_position,
        'candidate_count': len(ex.candidate_movie_ids),
        'true_positive_movie_id': ex.true_positive_movie_id,
    }
    for ex in examples
])
display(preview)


## 7. Select the available model variants


In [ ]:
configs = available_configs(
    ROOT,
    CHECKPOINT_DIR,
    base_model=BASE_MODEL,
    lora_model=LORA_MODEL,
)


def config_ready(cfg):
    model_path = Path(str(cfg['model_name']))
    if cfg.get('optional') and not model_path.exists():
        print(f"[skip] {cfg['label']} because {model_path} is missing")
        return False
    if cfg['mode'] == 'candidates':
        emb_path = Path(cfg['embedding_adapter_path'])
        proj_path = Path(cfg['projected_embeddings_path'])
        if not emb_path.exists() or not proj_path.exists():
            missing = emb_path if not emb_path.exists() else proj_path
            print(f"[skip] {cfg['label']} because {missing} is missing")
            return False
    return True

active_configs = [cfg for cfg in configs if config_ready(cfg)]
print('Active configs:')
for cfg in active_configs:
    print(f"  - {cfg['key']}: {cfg['label']} ({cfg['mode']}) -> {cfg['model_name']}")

if not active_configs:
    raise RuntimeError('No XAI configs are available. Check the model path and checkpoints.')


## 8. Run XAI and save results


In [ ]:
SEGMENT_ORDER = ('instruction', 'history', 'candidates', 'question', 'continuation', 'other')


def ensure_dir(path):
    path = Path(path)
    path.mkdir(parents=True, exist_ok=True)
    return path


def save_json(data, path):
    path = Path(path)
    ensure_dir(path.parent)
    path.write_text(json.dumps(data, indent=2), encoding='utf-8')


def load_json(path):
    return json.loads(Path(path).read_text(encoding='utf-8'))


def score_margin(value):
    if isinstance(value, dict):
        return float(value.get('score_margin', value.get('prob', 0.0)))
    return float(value)


def flatten_segment_columns(row, prefix, segment_scores):
    for segment in SEGMENT_ORDER:
        row[f'{prefix}:{segment}'] = float(segment_scores.get(segment, 0.0))


def flatten_result(result, cfg, target_mode, example):
    row = {
        'config_key': cfg['key'],
        'config_label': cfg['label'],
        'model_name': str(cfg['model_name']),
        'ranking_mode': cfg['mode'],
        'target_mode': target_mode,
        'user_id': int(example.user_id),
        'true_position': int(example.true_position),
        'target_label': result['selection']['target_label'],
        'top_candidate_label': result['selection']['top_candidate_label'],
        'target_candidate_rank': int(result['selection']['target_candidate_rank']),
        'positive_candidate_rank': int(result['selection']['positive_candidate_rank']),
        'top_candidate_rank': int(result['selection']['top_candidate_rank']),
        'top_score': score_margin(result['selection']['top_score']),
        'target_score': score_margin(result['selection']['target_score']),
        'prompt_token_count': int(result['prompt_token_count']),
    }

    flatten_segment_columns(row, 'IG', result['segment_summary'].get('integrated_gradients', {}))
    if result['segment_summary'].get('attention_rollout') is not None:
        flatten_segment_columns(row, 'ROLL', result['segment_summary'].get('attention_rollout', {}))
    flatten_segment_columns(row, 'ALTI', result['segment_summary'].get('alti_plus', {}))
    flatten_segment_columns(row, 'CAM', result['segment_summary'].get('grad_cam', {}))
    return row


def flatten_layer_rows(result, cfg, target_mode, example):
    rows = []
    for layer_item in result['layer_attributions']:
        rows.append({
            'config_key': cfg['key'],
            'config_label': cfg['label'],
            'model_name': str(cfg['model_name']),
            'ranking_mode': cfg['mode'],
            'target_mode': target_mode,
            'user_id': int(example.user_id),
            'layer_index': int(layer_item['layer_index']),
            'layer_score': float(layer_item['layer_score']),
            'attribution_method': str(layer_item.get('attribution_method', '')),
            'attribution_mode': str(layer_item.get('attribution_mode', '')),
        })
    return rows


def record_path(cfg, example, target_mode):
    return RUN_ROOT / 'records' / cfg['key'] / target_mode / f'user_{example.user_id}.json'


def run_config(cfg):
    print()
    print('=' * 88)
    print(f"{cfg['label']} | {cfg['mode']} | {cfg['model_name']}")
    print('=' * 88)

    pipeline = RankingXAIPipeline(
        model_name=cfg['model_name'],
        checkpoint_dir=CHECKPOINT_DIR,
        device=DEVICE,
        freeze_llm=True,
        train_adapter=False,
        load_embedding_adapter=bool(cfg.get('load_embedding_adapter', False)),
        embedding_adapter_path=cfg.get('embedding_adapter_path'),
        projected_embeddings_path=cfg.get('projected_embeddings_path'),
    )

    cfg_dir = ensure_dir(RUN_ROOT / cfg['key'])
    rows = []
    for example in examples:
        for target_mode in TARGET_MODES:
            print(f"  [run] user={example.user_id} target_mode={target_mode}")
            try:
                result = pipeline.analyze_example(
                    example,
                    mode=cfg['mode'],
                    target_mode=target_mode,
                    ig_steps=IG_STEPS,
                    layer_steps=LAYER_STEPS,
                    layer_attribution_method=LAYER_ATTRIBUTION_METHOD,
                    include_attention_maps=INCLUDE_ATTENTION_MAPS,
                    include_metrics=False,
                )
            except Exception as exc:
                print(f"    [error] user={example.user_id} target_mode={target_mode}: {exc}")
                continue

            out_path = record_path(cfg, example, target_mode)
            save_json(result, out_path)
            rows.append(flatten_result(result, cfg, target_mode, example))
            print(f"    [saved] {out_path}")

    summary_path = cfg_dir / 'summary.csv'
    pd.DataFrame(rows).to_csv(summary_path, index=False)
    print(f"[done] summary -> {summary_path}")
    return rows


all_rows = []
for cfg in active_configs:
    try:
        all_rows.extend(run_config(cfg))
    except Exception as exc:
        print(f"[error] config {cfg['label']} failed: {exc}")

analysis_df = pd.DataFrame(all_rows)
analysis_df.to_csv(RUN_ROOT / 'summary_all.csv', index=False)
print()
print('[done] consolidated summary ->', RUN_ROOT / 'summary_all.csv')
display(analysis_df)


## 9. Quick summary


In [ ]:
if analysis_df.empty:
    print('No XAI rows were produced.')
else:
    cols = [
        'config_label',
        'ranking_mode',
        'target_mode',
        'user_id',
        'target_label',
        'top_candidate_label',
        'target_candidate_rank',
        'positive_candidate_rank',
        'top_score',
        'target_score',
    ]
    for prefix in ('IG', 'ROLL', 'ALTI', 'CAM'):
        for segment in SEGMENT_ORDER:
            key = f'{prefix}:{segment}'
            if key in analysis_df.columns:
                cols.append(key)
    display(analysis_df[cols])


## 10. Optional next steps

- Increase `SAMPLE_SIZE` if you want more than one user.
- Drop `model_choice` from `TARGET_MODES` if you only want the positive candidate.
- Set `INCLUDE_ATTENTION_MAPS = False` if you want a faster pass.
